# Week 1, Day 3: CASE WHEN

Exercises: `day-03.md`

In [1]:
import sys
sys.path.append("../..")

import numpy as np
import pandas as pd
import plotly.express as px

from db import connect

con = connect()

## 3.1

In [12]:
con.sql("""
select
payment_type
,count(distinct order_id) as orders_count
,round(sum(payment_value),2) as payment_value_sum
from order_payments
group by all
order by orders_count desc
""").df()

,payment_type,orders_count,payment_value_sum
0,credit_card,76505,12542084.19
1,boleto,19784,2869361.27
2,voucher,3866,379436.87
3,debit_card,1528,217989.79
4,not_defined,3,0.00


## 3.2

In [28]:
con.sql("""
select
--1x, 2-3x, 4-6x, 7x+
case
    when payment_installments == 1 then '1'
    when payment_installments between 2 and 3 then '2-3x'
    when payment_installments between 4 and 6 then '4-6x'
    when payment_installments > 6 then '7x+'
    else 'other'
end as payment_installments_group
,count(order_id) as orders_count
,round(count(order_id) / sum(count(*)) over () * 100,1) as pct_orders
from order_payments
group by all
order by orders_count desc
""").df()

,payment_installments_group,orders_count,pct_orders
0,1,52546,50.6
1,2-3x,22874,22.0
2,4-6x,16257,15.6
3,7x+,12207,11.8
4,other,2,0.0


## 3.3

In [58]:
con.sql("""
select
sum( case
    when order_status in ('delivered') and order_delivered_customer_date is not null then 1
    else 0
end) as 'total_orders'
,sum( case
    when order_status in ('delivered') and order_delivered_customer_date > order_estimated_delivery_date then 1
    else 0
end) as 'late_orders'
,round(sum( case
    when order_status in ('delivered') and order_delivered_customer_date > order_estimated_delivery_date then 1
    else 0
end) / sum( case
    when order_status in ('delivered') and order_delivered_customer_date is not null then 1
    else 0
end) * 100,1) as 'pct_orders'
from orders
group by all
""").df()

,total_orders,late_orders,pct_orders
0,96470.0,7826.0,8.1


## 3.4

In [86]:
df1 = con.sql("""
select * from order_payments
""").df()

#1x, 2-3x, 4-6x, 7x+

df1['bins'] = pd.cut(df1['payment_installments'], bins=[0,1,2,3,4,6,1000])

df1['bins'].value_counts(normalize=True)

bins
(0, 1]       0.505814
(1, 2]       0.119489
(6, 1000]    0.117506
(2, 3]       0.100699
(4, 6]       0.088166
(3, 4]       0.068326
Name: proportion, dtype: float64

## 3.5

In [107]:
df2.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,month_start,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017-10,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018-08,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018-08,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,2017-12,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2018-02,0


In [112]:
df2 = con.sql("""
select * from orders
""").df()

delivered = df2[(df2['order_status'].isin(['delivered'])) & df2['order_delivered_customer_date'].notna()].copy()

delivered['month_start'] = pd.to_datetime(delivered['order_purchase_timestamp']).dt.to_period('M')

delivered['is_late'] = np.where(delivered['order_delivered_customer_date'] > delivered['order_estimated_delivery_date'], 1, 0)

delivered_agg = delivered.groupby(['month_start','is_late'],as_index=False).agg(orders_count=('order_id','count'))

delivered_pivot = delivered_agg.pivot(index='month_start', columns='is_late', values='orders_count').fillna(0)

delivered_pivot['total_orders'] = delivered_pivot[0] + delivered_pivot[1] 

delivered_pivot['late_share'] = round(delivered_pivot[1] / delivered_pivot['total_orders'] * 100,1)

delivered_pivot

is_late,0,1,total_orders,late_share
month_start,,,,
2016-09,0.0,1.0,1.0,100.0
2016-10,262.0,3.0,265.0,1.1
2016-12,1.0,0.0,1.0,0.0
2017-01,727.0,23.0,750.0,3.1
2017-02,1600.0,53.0,1653.0,3.2
2017-03,2404.0,142.0,2546.0,5.6
2017-04,2122.0,181.0,2303.0,7.9
2017-05,3417.0,128.0,3545.0,3.6
2017-06,3014.0,121.0,3135.0,3.9


In [111]:
delivered = df2[(df2["order_status"] == "delivered") & df2["order_delivered_customer_date"].notna()].copy()
delivered["month"] = delivered["order_purchase_timestamp"].dt.to_period("M")
delivered["is_late"] = np.where(delivered["order_delivered_customer_date"] > delivered["order_estimated_delivery_date"], 1, 0)
(delivered[delivered["month"] >= "2018-01"].groupby("month")["is_late"].mean() * 100).round(1)

month
2018-01     6.6
2018-02    16.0
2018-03    21.4
2018-04     5.3
2018-05     8.2
2018-06     1.4
2018-07     4.5
2018-08    10.4
Freq: M, Name: is_late, dtype: float64